In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import re

In [4]:
first_round = pd.read_csv("datos/classification first round/classification_final.csv")
second_round = pd.read_csv("datos/classification second_round/classification_secondRound_final.csv")

classifiers_first = pd.read_csv("datos/classification first round/classifiers_final.csv")
classifiers_second = pd.read_csv("datos/classification second_round/classifiers_secondRound_final.csv")

In [5]:
file_path = "_tempavisos.dta"

chunk_size = 10000
chunks = pd.read_stata(file_path, columns=['avisoid', 'avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo'], chunksize=chunk_size)

avisos = pd.DataFrame()

for chunk in chunks:
    avisos = pd.concat([avisos, chunk], ignore_index=True)

In [6]:
avisos = pd.merge(first_round, avisos, left_on='ad_id', right_on='avisoid', how='left')

In [8]:
avisos['aviso'] = avisos[['avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo']].apply(
    lambda row: ' '.join(row.dropna().astype(str)), axis=1
)

In [10]:
# Function to clean HTML tags
def remove_html_tags(text):
    if pd.isnull(text):
        return ""
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

# Clean and count words
avisos['aviso'] = avisos['aviso'].apply(remove_html_tags)

## Conservative approach

We take down all of the dissagreement

In [16]:
# Step 1: Group by 'ad_id' and collect unique classifications
classification_counts = avisos.groupby('ad_id')['classification'].nunique()

# Step 2: Filter to keep only ad_ids with a single unique classification (i.e., same classification twice)
valid_ad_ids = classification_counts[classification_counts == 1].index

# Step 3: Filter the original DataFrame to keep only those ad_ids
avisos_filtered = avisos[avisos['ad_id'].isin(valid_ad_ids)]


In [23]:
avisos_filtered = avisos_filtered.drop_duplicates(subset=['aviso', 'classification'])

In [26]:
train_data = avisos_filtered[['aviso', 'classification']]

In [28]:
train_data.to_csv("datos/training/conservative_td.csv", index=False, encoding='utf-8')